# GRL cover image

Generates the cover image submitted with Sauthoff and others (2026), *Geophysical Research Letters*: the
forward-filled evolving outlines of the Mercer and Whillans ice stream lakes (Lower Conway, Lower Mercer,
Conway, Mercer and Upper Conway subglacial lakes) draped on the mean CryoSat-2/ICESat-2 surface with MODIS
Mosaic of Antarctica 2014 imagery, above a floor showing each pixel's lakebed active area frequency,
normalized per lake.

Inputs:
* `output/lake_outlines/evolving_outlines/forward_fill/` and `output/cycle_dates.csv` (this repository)
* `DATA_DIR/ATL14_A3_0325_100m_004_05.nc`: ICESat-2 ATL14 Antarctic Land Ice Height, Version 4 (doi:10.5067/ATLAS/ATL14.004), quadrant A3. Its 100 m DEM height `h` is the terrain surface.
* `DATA_DIR/moa125_2014_hp1_v01.tif`: MODIS Mosaic of Antarctica 2014, 125 m (NSIDC-0730, https://nsidc.org/data/nsidc-0730/versions/1)

The submitted image used a time-mean CryoSat-2/ICESat-2 surface height (`CryoSat2_ICESat2_h_dh.nc`, not archived). From v1.1 the surface is ATL14, a public data set. Compared with the submitted image at full resolution (6858 × 6858 pixels), 0.93% of pixels differ by more than 10 grey levels, because the terrain's appearance comes mostly from the MOA imagery draped over it.

# Setup computing environment

In [ ]:
# Import packages
import geopandas as gpd
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os
import pandas as pd
import rasterio.enums
import rasterio.features
from rasterio.transform import Affine
import rioxarray
from scipy.interpolate import RegularGridInterpolator
import xarray as xr

# Magic functions
%matplotlib widget

# Define data and output directories dependent on home environment
if os.getenv('HOME') == '/home/jovyan':
    DATA_DIR = '/home/jovyan/data'
    OUTPUT_DIR = '/home/jovyan/1_evolving_lakes/output'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9,
    'axes.labelsize': 18,
    # 'axes.titlesize': 9,
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    # 'legend.fontsize': 12,
})

# Import data

In [ ]:
# Import cycle dates, which set the limits of the outline time color scale
cyc_datetimes = pd.read_csv('output/cycle_dates.csv', parse_dates=['cyc_start_datetimes', 'cyc_end_datetimes'])
cyc_start_datetimes = [np.datetime64(ts) for ts in cyc_datetimes['cyc_start_datetimes']]
cyc_end_datetimes = [np.datetime64(ts) for ts in cyc_datetimes['cyc_end_datetimes']]

# View dates
cyc_datetimes.head()

# Figure

In [ ]:
# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================

# VISUAL QUALITY
UPSAMPLE_FACTOR = 1        # ATL14 is already 100 m; the earlier 1 km surface was refined 10x to the same grid
PIXEL_SIZE_DENSITY = 250   # Density map resolution
MOA_CLIM = [14000, 17000] 

# AESTHETICS
CMAP_OUTLINES = 'plasma'   
CMAP_DENSITY = 'Blues'
LINE_WIDTH = 1.5
FLOAT_OFFSET = 1.0        
ELEVATION = 30
AZIMUTH = -45
BOX_ASPECT_Z = 0.7        # Height of the plot box
GAP_RATIO = 10            # Vertical gap size

# --- COLORBAR LIMITS ---
TIME_MIN = cyc_start_datetimes[0]
TIME_MAX = cyc_end_datetimes[-1]
DENS_MIN = 0
DENS_MAX = 100

# --- NEW: MULTI-LAKE CONFIGURATION ---
# Directory containing the lake GeoJSONs
LAKE_OUTLINE_DIR = 'output/lake_outlines/evolving_outlines/forward_fill'

# List of filenames (without .geojson extension if you prefer, or adjust loop)
LAKE_NAMES = [
    'LowerConwaySubglacialLake', 
    'LowerMercerSubglacialLake',
    'ConwaySubglacialLake', 
    'MercerSubglacialLake',
    'UpperSubglacialLakeConway', 
    # 'Thw_70', 'Thw_124', 'Thw_142', 'Thw_170'
    # 'Mac1', 'Mac2', 'Mac3', 'Mac4', 
]

ATL14_PATH = DATA_DIR + '/ATL14_A3_0325_100m_004_05.nc'  # ATL14 v004, quadrant A3
MOA_PATH = DATA_DIR + '/moa125_2014_hp1_v01.tif'

# ==========================================
# 2. DATA LOADING & PREP
# ==========================================
print("Loading Data...")

# Load Background Data
moa_da = rioxarray.open_rasterio(MOA_PATH).squeeze()

# Dictionary to store individual lake dataframes for density calculation later
lake_data_store = {} 
all_gdfs_list = []

for lake_name in LAKE_NAMES:
    full_path = os.path.join(LAKE_OUTLINE_DIR, f"{lake_name}.geojson")
    
    if os.path.exists(full_path):
        print(f"Loading: {lake_name}")
        temp_gdf = gpd.read_file(full_path)
        
        # Ensure consistent CRS
        if temp_gdf.crs is None: 
            temp_gdf.set_crs("EPSG:3031", inplace=True)
        temp_gdf = temp_gdf.to_crs("EPSG:3031")
        
        # Parse Dates
        temp_gdf['mid_pt_datetime'] = pd.to_datetime(temp_gdf['mid_pt_datetime'])
        temp_gdf = temp_gdf.sort_values('mid_pt_datetime')
        
        # Store for combined bounds AND individual processing
        all_gdfs_list.append(temp_gdf)
        lake_data_store[lake_name] = temp_gdf
    else:
        print(f"WARNING: File not found for {lake_name} at {full_path}")

if not all_gdfs_list:
    raise ValueError("No valid lake files were loaded.")

# Master GDF for bounds and plotting lines
gdf_outlines = pd.concat(all_gdfs_list, ignore_index=True)

# Generate date numbers for the master timeline (for plotting lines)
date_nums = mdates.date2num(gdf_outlines['mid_pt_datetime'])

# Handle Custom Time Limits
if TIME_MIN: t_min_num = mdates.date2num(pd.to_datetime(TIME_MIN))
else: t_min_num = date_nums.min()

if TIME_MAX: t_max_num = mdates.date2num(pd.to_datetime(TIME_MAX))
else: t_max_num = date_nums.max()

# ==========================================
# 3. GENERATE PER-LAKE NORMALIZED DENSITY
# ==========================================
print("Calculating Per-Lake Normalized Probability Density...")

# 1. Setup the Global Grid (based on ALL lakes)
minx, miny, maxx, maxy = gdf_outlines.total_bounds
pad = 3000
minx -= pad; maxx += pad; miny -= pad; maxy += pad

minx = np.floor(minx / PIXEL_SIZE_DENSITY) * PIXEL_SIZE_DENSITY
maxx = np.ceil(maxx / PIXEL_SIZE_DENSITY) * PIXEL_SIZE_DENSITY
miny = np.floor(miny / PIXEL_SIZE_DENSITY) * PIXEL_SIZE_DENSITY
maxy = np.ceil(maxy / PIXEL_SIZE_DENSITY) * PIXEL_SIZE_DENSITY

width = int((maxx - minx) / PIXEL_SIZE_DENSITY)
height = int((maxy - miny) / PIXEL_SIZE_DENSITY)
transform = Affine.translation(minx, miny) * Affine.scale(PIXEL_SIZE_DENSITY, PIXEL_SIZE_DENSITY)

# Initialize the final composite grid
final_density_grid = np.zeros((height, width), dtype=np.float32)

# 2. Process Each Lake Individually
for lake_name, lake_gdf in lake_data_store.items():
    
    # Create a temporary grid just for this lake
    lake_grid_acc = np.zeros((height, width), dtype=np.float32)
    
    # Group by date to count unique observations for THIS lake
    grouped = lake_gdf.groupby('mid_pt_datetime')
    lake_timesteps = len(grouped)
    
    if lake_timesteps == 0: continue

    # Rasterize this lake's history
    for date, group in grouped:
        daily_mask = rasterio.features.rasterize(
            [(geom, 1) for geom in group.geometry],
            out_shape=(height, width), transform=transform, fill=0, dtype=np.uint8,
            merge_alg=rasterio.enums.MergeAlg.replace 
        )
        lake_grid_acc += daily_mask
        
    # Normalize: (Pixels Active / Total Dates for THIS lake) * 100
    lake_pct_grid = (lake_grid_acc / lake_timesteps) * 100
    
    # Add to master grid
    # We use np.maximum so that if lakes overlap, we show the highest probability 
    # (active 100% of the time in Lake A OR Lake B is still 100% active)
    final_density_grid = np.maximum(final_density_grid, lake_pct_grid)

# Finalize variables for plotting
density_pct = final_density_grid
density_x = np.linspace(minx, maxx, width)
density_y = np.linspace(miny, maxy, height)
dX, dY = np.meshgrid(density_x, density_y)

# ==========================================
# 4. HIGH-RES SURFACE GENERATION
# ==========================================
# ATL14 DEM height, cropped to the plot extent plus 1 km so the interpolation below has data at the edges
Z_data = xr.open_dataset(ATL14_PATH)['h'].sel(x=slice(minx - 1000, maxx + 1000),
                                               y=slice(miny - 1000, maxy + 1000))

print(f"Upsampling Terrain ({UPSAMPLE_FACTOR}x)...")
orig_res = abs(Z_data.x[1] - Z_data.x[0])
new_res = orig_res / UPSAMPLE_FACTOR
num_x = int(abs(maxx - minx) / new_res)
num_y = int(abs(maxy - miny) / new_res)

# Generate new grid coordinates
new_x = np.linspace(minx, maxx, num_x)
if Z_data.y[0] > Z_data.y[-1]: 
    new_y = np.linspace(maxy, miny, num_y) # Descending Y (standard for map projections)
else: 
    new_y = np.linspace(miny, maxy, num_y)

# Interpolate DEM to new grid
surf_highres = Z_data.interp(x=new_x, y=new_y, method='linear')

# Interpolate Imagery (MOA) to new grid
# Subset first to save memory before interpolation
moa_subset = moa_da.sel(x=slice(minx-5000, maxx+5000), y=slice(maxy+5000, miny-5000))

if moa_subset.size == 0:
    print("Warning: MOA Imagery subset is empty. Check coordinate bounds.")
    # Fallback to zeros if MOA is missing/out of bounds
    moa_remapped = xr.DataArray(np.zeros((num_y, num_x)), coords={'y': new_y, 'x': new_x})
else:
    moa_remapped = moa_subset.interp(x=new_x, y=new_y, method='linear')

norm_img = Normalize(vmin=MOA_CLIM[0], vmax=MOA_CLIM[1])
face_colors = cm.gray(norm_img(moa_remapped.values))

# ==========================================
# 5. PLOTTING
# ==========================================
fig = plt.figure(figsize=(14, 12), dpi=300)
ax = fig.add_subplot(111, projection='3d')

X_grid, Y_grid = np.meshgrid(surf_highres.x, surf_highres.y)
Z_grid = surf_highres.values

z_min_ice = np.nanmin(Z_grid)
z_max_ice = np.nanmax(Z_grid)
ice_relief = z_max_ice - z_min_ice
if ice_relief < 1: ice_relief = 50 
z_floor = z_min_ice - (ice_relief * GAP_RATIO)

# --- A. Plot Probability Floor ---
density_plot = np.where(density_pct < 1, np.nan, density_pct)
cset = ax.contourf(dX, dY, density_plot, zdir='z', offset=z_floor, 
                   levels=np.linspace(DENS_MIN, DENS_MAX, 20),
                   cmap=CMAP_DENSITY, alpha=0.8)

# --- B. Plot Surface ---
surf = ax.plot_surface(X_grid, Y_grid, Z_grid, facecolors=face_colors,
                        rstride=1, cstride=1, linewidth=0, antialiased=False, shade=True)

# --- C. Plot Draped Outlines ---
print("Draping Outlines...")
interp_x = surf_highres.x.values
interp_y = surf_highres.y.values
interp_z = surf_highres.values

# Handle sorting for Interpolator requirements
if interp_x[0] > interp_x[-1]: 
    interp_x = interp_x[::-1]; interp_z = np.flip(interp_z, axis=1)
if interp_y[0] > interp_y[-1]: 
    interp_y = interp_y[::-1]; interp_z = np.flip(interp_z, axis=0)
    
interp_func = RegularGridInterpolator((interp_y, interp_x), interp_z, bounds_error=False, fill_value=np.nan)

norm_time = Normalize(vmin=t_min_num, vmax=t_max_num)
cmap_time = plt.get_cmap(CMAP_OUTLINES)

# Loop through the combined DataFrame containing all lakes
for i, row in enumerate(gdf_outlines.itertuples()):
    poly = row.geometry
    current_date_num = date_nums[i]
    color_val = cmap_time(norm_time(current_date_num))
    
    if poly.geom_type == 'Polygon':
        x_poly, y_poly = poly.exterior.xy
        pts = np.vstack((np.array(y_poly), np.array(x_poly))).T 
        z_poly = interp_func(pts) + FLOAT_OFFSET
        ax.plot(np.array(x_poly), np.array(y_poly), z_poly, 
                color=color_val, linewidth=LINE_WIDTH, zorder=10)
    elif poly.geom_type == 'MultiPolygon':
        for sub_poly in poly.geoms:
            x_poly, y_poly = sub_poly.exterior.xy
            pts = np.vstack((np.array(y_poly), np.array(x_poly))).T 
            z_poly = interp_func(pts) + FLOAT_OFFSET
            ax.plot(np.array(x_poly), np.array(y_poly), z_poly, 
                    color=color_val, linewidth=LINE_WIDTH, zorder=10)

# ==========================================
# 6. FINAL STYLING
# ==========================================
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
z_visual_bottom = z_floor - (ice_relief * 0.2) 
z_visual_top = z_max_ice + (ice_relief * 0.5)
ax.set_zlim(z_visual_bottom, z_visual_top)
ax.set_box_aspect((1, 1, BOX_ASPECT_Z))
ax.view_init(elev=ELEVATION, azim=AZIMUTH)
ax.set_axis_off()

# --- COLORBAR 1: TIME ---
# sm_time = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
# sm_time.set_array([])
# cbar_t = plt.colorbar(sm_time, ax=ax, location='top', orientation='horizontal', shrink=0.4, aspect=30, pad=-0.9)
# cbar_t.ax.xaxis.set_major_locator(mdates.YearLocator(1))
# cbar_t.ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
# cbar_t.set_label('Evolving Lake Outline Observation', labelpad=5)
sm_time = cm.ScalarMappable(cmap=cmap_time, norm=norm_time)
sm_time.set_array([])
cbar_t = plt.colorbar(sm_time, ax=ax, location='top', orientation='horizontal', shrink=0.4, aspect=30, pad=-0.09)

# 1. MAJOR TICKS: Force a tick for every single year
cbar_t.ax.xaxis.set_major_locator(mdates.YearLocator(1))

# 2. LABELS: Manually hide every other label (keep even years like 2014, 2016...)
# This mimics the "Auto" spacing you liked while keeping the physical ticks for every year.
def sparse_year_formatter(x, pos):
    dt = mdates.num2date(x)
    if dt.year % 2 == 0: 
        return dt.strftime('%Y')
    else: 
        return ''
        
cbar_t.ax.xaxis.set_major_formatter(ticker.FuncFormatter(sparse_year_formatter))

# 3. MINOR TICKS: Add ticks for every quarter (Jan, Apr, Jul, Oct)
cbar_t.ax.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))

cbar_t.set_label('Evolving Lake Outline Observation', labelpad=10)

# --- COLORBAR 2: DENSITY ---
norm_dens = Normalize(vmin=DENS_MIN, vmax=DENS_MAX)
sm_dens = cm.ScalarMappable(cmap=CMAP_DENSITY, norm=norm_dens)
sm_dens.set_array([])
cbar_d = plt.colorbar(sm_dens, ax=ax, location='bottom', orientation='horizontal', shrink=0.4, aspect=30, pad=-0.25)
cbar_d.set_label('Lakebed Active Area Frequency (%)', labelpad=5)

plt.tight_layout()

# Save before showing: with a non-interactive backend, plt.show() closes the figure
save_path = OUTPUT_DIR + '/figures/GRL71776_cover_image_Mercer_Whillans.png'
# save_path = 'output/GRL71776_cover_image_MacAyeal.png'
plt.savefig(save_path, dpi=600, bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
plt.close('all')